# World session 4 · Finding the names

**Goal:** find every personal detail in 200,000 incident reports, so they can be redacted. Course 1's sessions 6 and 7 found meaning in short notes. These reports name people on purpose: names, SIDs, birth dates and witnesses' phone numbers, written every way people write them. A **span** is where a detail sits in its report: the report's id, and the characters it starts and ends at (counted from 0, end excluded, as Python slices them).

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Your answers are checked against the world's **answer keys**: the generator's own record of every mess it made. A result you hand in (rows you found) is graded by **precision** (how many of yours are right) and **recall** (how many of the right ones you found).
- Nothing here can change the lake. You read the messy files as a source system delivered them.

> ⚠️ **Mock data only.** Every person, place and record in this world is invented.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and opens the world's messy files.
SESSION = "4"

import sys
from pathlib import Path

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "lakehouse" / "sessions" / "check.py").exists():
        sys.path.insert(0, str(folder))
        break

import re
import pandas as pd
from lakehouse.sessions import check
from lakehouse.sessions.check import ___

con = check.world()       # DuckDB: one view messy_<table> per messy CSV folder, every column as text
print(f"World session {SESSION} ready. Messy tables:", ", ".join(r[0] for r in con.sql("SHOW TABLES").fetchall()))

## 1. The reports

In [ ]:
reports = con.sql("SELECT CAST(incident_report_id AS BIGINT) AS incident_report_id, report_kind, narrative FROM messy_incident_report WHERE narrative IS NOT NULL").df()
for text in reports.narrative.sample(3, random_state=4):
    print(text, "\n")

Three kinds: a hero's **field report**, a civilian's **witness statement** (with their own name and phone), and the **desk report** of the call.

## 2. A span finder
`spans(pattern)` runs a regular expression over every report and returns one row per match: `incident_report_id`, `start`, `end`. With `group=1` the span is the first bracketed part of the pattern instead of the whole match.

In [ ]:
def spans(pattern, group=0):
    """One row per match of pattern in every report: its id, and where the match (or a group) starts and ends."""
    rx = re.compile(pattern)
    rows = [(i, m.start(group), m.end(group)) for i, text in zip(reports.incident_report_id, reports.narrative)
            for m in rx.finditer(text) if m.group(group)]
    return pd.DataFrame(rows, columns=["incident_report_id", "start", "end"])

def show(found, n=5):
    """A few of the found spans in their context."""
    text = dict(zip(reports.incident_report_id, reports.narrative))
    for i, s, e in found.head(n).itertuples(index=False):
        print(repr(text[i][max(0, s - 20):e + 10]), "->", repr(text[i][s:e]))

## 3. Numbers with a shape
SIDs and phone numbers have a fixed shape, so a regular expression can find them all. Look at how they're written first.

In [ ]:
show(spans(r"\b(?:SID|sid)\b.{0,5}\d{7}"), 8)

🧪 **Your turn #1.** Build `sids`: every SID, label and number together (`SID# 4000034`, `S.I.D. 4000401`, ...). The bar is 99% precision and recall.

In [ ]:
sids = ___      # spans(r"...")
check.w4_ex1(sids)

🧪 **Your turn #2.** Build `phones`: every phone number. The world's numbers are fictional (`555-01xx`), written three ways.

In [ ]:
phones = ___      # spans(r"...")
check.w4_ex2(phones)

## 4. Birth dates
A birth date follows `DOB `, in any of the date formats from world session 2. Capture only the date, as group 1.

🧪 **Your turn #3.** Build `birth_dates`. The bar is 99% precision and 90% recall.

In [ ]:
birth_dates = ___      # spans(r"DOB (...)", group=1)
check.w4_ex3(birth_dates)

## 5. Names: no shape to find
A name has no fixed shape: `M. Flores`, `SAMANTHA TUCKER`, `BRADLEY, LUIS`, `Cynthia Reynolds`. What gives it away is the **context**: the words before it. And once a report names someone, it often names them again (`M. Flores was taken into custody`).

In [ ]:
NAME = r"((?:[A-Z]\.|[A-Z][A-Za-z'\-]+)(?:,? [A-Z][A-Za-z'\-]+))"
first_try = spans(r"(?i:identified as) " + NAME, group=1)
show(first_try)
print(len(first_try), "found")

🧪 **Your turn #4.** Build `names`: every person's name. Two steps:
1. Find names by the words before them: `identified as`, `my name is`, `records check:`, `bystander`, `victim`, `witness`, or a name alone in brackets.
2. In each report, find every other place the same name appears.

The bar is 95% precision and 85% recall. Names are the hardest part of redaction; real tools miss some too.

In [ ]:
names = ___      # step 1 and step 2, as one DataFrame of spans
check.w4_ex4(names)

## 6. How the lake does it
The lake keeps the raw reports in silver, where only engineering can read them, and builds `world_gold.incident_report_redacted`, every detail replaced by its kind. That's what the MCP server's `search_incident_reports` searches. The lab redacts by the world's own key; a real platform would run a finder like yours, and would keep treating the text as personal, because a finder misses things.

In [ ]:
from lakehouse import lake
silver = lake.lake_ro_connect()
print(silver.sql("SELECT narrative FROM lake.world_gold.incident_report_redacted WHERE report_kind = 'witness statement' LIMIT 1").fetchone()[0])

## Wrap-up
- Shaped details (SIDs, phones, dates) fall to a regular expression once you've seen every way they're written.
- Names have no shape: find them by context, then find their repeats.
- Measure a finder with precision and recall against known answers before trusting it.
- Keep treating redacted free text as personal: a finder that's right 95% of the time still misses some.

## Answers
Try each exercise first. Each answer replaces the `___` line in its 🧪 cell.

**#1** The label in its seven spellings, then the seven digits.

```python
sids = spans(r"(?:S\.I\.D\.|SID|sid)(?:#| no\.| NO\.|:)? ?\d{7}")
```

**#2** Three formats around the fictional 555-01xx.

```python
phones = spans(r"(?:\(\d{3}\) |\b\d{3}[-.])555[-.]01\d\d\b")
```

**#3** After DOB: a numeric date with / or -, a month name date, or 01-MAR-1936.

```python
birth_dates = spans(r"DOB (\d{1,4}[-/][0-9A-Za-z]{1,3}[-/]\d{2,4}|[A-Z][a-z]+ \d{1,2}, \d{4})", group=1)
```

**#4** Context first, then every repeat of each name found in the same report.

```python
names = pd.DataFrame(
    [(i, m.start(), m.end())
     for i, text in zip(reports.incident_report_id, reports.narrative)
     for name in {m.group(1) or m.group(2) for m in re.finditer(
         r"(?i:identified as|my name is|records check:|bystander|victim|witness) " + NAME + r"|\(" + NAME + r"\)", text)}
     for m in re.finditer(re.escape(name), text)],
    columns=["incident_report_id", "start", "end"]).drop_duplicates()
```